# ENCflow をブラウザで動かす(インストール不要)

このノートブックは、洪水・流域水文シミュレーションプログラム
**[ENCflow](https://github.com/ENCflow/ENCflow)** を、
パソコンに何もインストールせずに Google Colab 上で動かす入門です。

- 上から順にセルを実行するだけです(セル左端の ▶ を押すか、`Shift+Enter`)。
- 所要時間は全部で 5 分ほどです。
- セルの中の `!` や `%%bash` で始まる行は、Linux のコマンドをそのまま
  実行しています。将来自分のパソコン(WSL 等)で動かすときも同じコマンドが使えます
  ([Windows での使い方](https://github.com/ENCflow/ENCflow/blob/main/docs/windows.md))。


## 1. 準備: コンパイルして実行ファイルを作る

ENCflow は外部ライブラリゼロの Fortran プログラムなので、
コンパイラ(gfortran)さえあればどこでもビルドできます。1〜2 分かかります。


In [ ]:
%%bash
# Fortran コンパイラを用意し、ENCflow を取得してビルドする
apt-get -qq install -y gfortran > /dev/null
git clone --depth 1 -q https://github.com/ENCflow/ENCflow.git
cd ENCflow/src && make install -j2 2>&1 | tail -2


## 2. 最初の計算: 水の山が崩れて広がる

最小の例題(test/wave)を実行します。静かな水面に立てた「水の山」が
崩れて同心円状に広がるだけの計算です。`Run.sh` は実行後に、リポジトリに
収録された基準結果と**ビット単位で一致するか**を自動照合します —
`PASS` が出れば、あなたの環境で開発者と同じ答えが得られたということです。


In [ ]:
%%bash
cd ENCflow/test/wave && ./Run.sh 2>&1 | tail -3


## 3. 結果を見る

出力はただのテキスト行列なので、numpy で読んで matplotlib で描くだけです。
`E0000.txt` … `E0008.txt` が 1 秒ごとの水位スナップショットです。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for k, ax in enumerate(axes.flat):
    e = np.loadtxt(f'ENCflow/test/wave/result/E{k:04d}.txt')
    im = ax.imshow(e, cmap='viridis', vmin=0.9, vmax=1.3)
    ax.set_title(f't = {k} s'); ax.axis('off')
fig.colorbar(im, ax=axes, shrink=0.6, label='water level (m)')
plt.show()


## 4. 自分の計算を作る: すり鉢地形に雨を降らせる

ENCflow の入力は**テキストファイルだけ**です。ここでは Python で
「すり鉢状の地形」と「30 分間 100 mm/h の豪雨」の設定を書き出して、
窪地が湛水していく様子を計算してみます。

パラメータファイルの各行の意味はコメントのとおりです。数値を変えて
再実行すれば、それがもう自分の数値実験です(例: 雨を 200 mm/h に、
地形をもっと深く、粗度 rn0 を大きく…)。


In [ ]:
import numpy as np

# --- 地形: すり鉢(中心 0 m、縁に向かって高くなる)---
n = 51
c = (n - 1) / 2
y, x = np.mgrid[0:n, 0:n]
z = 3.0 * ((x - c)**2 + (y - c)**2) / c**2
np.savetxt('z.txt', z, fmt='%.4f')

# --- パラメータファイル(これが ENCflow への入力のすべて)---
param = '''
! 降雨ですり鉢地形が湛水していく最小例
&list_sysparam
  dt = 0.1                  ! 時間刻み (s)
  tt = 1800                 ! 計算終了時刻 (s) = 30 分
  dt_disp = 180             ! 画面表示間隔 (s)
  dt_file = 180             ! ファイル出力間隔 (s)
  fn_geoinfo = '-'          ! '-' = このファイルから読む
  fn_initial = '-'
  fn_precip = '-'           ! 降雨機能はこの 1 行+下のグループで有効化
  dir_data = '.'            ! 入力データ(z.txt)の場所
  dir_result = 'result_rain'
/

&list_initial
  f_htype = 0               ! 初期水深: 固定値
  h0 = 0.0                  ! 乾いた状態から
/

&list_geoinfo
  lx = 102.0                ! 領域サイズ (m)
  ly = 102.0
  nx = 51                   ! 格子数
  ny = 51
  f_ztype = 1               ! 地盤高: ファイルから
  fn_z = 'z.txt'
  f_rntype = 0              ! 粗度係数: 固定値
  rn0 = 0.05
/

&list_precip
  prtype = 1                ! 一様降雨の時系列
  prval(1:2,1) = 0, 100     ! (分, mm/h): 0 分から 100 mm/h
  prval(1:2,2) = 9999, 100
/
'''
with open('param_rain.txt', 'w') as f:
    f.write(param)
print('z.txt と param_rain.txt を書き出しました')


In [ ]:
%%bash
# 実行(数十秒)。S(m) 列 = 領域平均の水量が雨で増えていくのが見える
./ENCflow/bin/encflow param_rain.txt | tail -5


## 5. 湛水のアニメーション


In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

frames = [np.loadtxt(f) for f in sorted(glob.glob('result_rain/H0*.txt'))]
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(frames[0], cmap='Blues', vmin=0, vmax=0.65)
ax.axis('off'); fig.colorbar(im, shrink=0.8, label='depth (m)')
ttl = ax.set_title('t = 0 min')
def update(k):
    im.set_data(frames[k]); ttl.set_text(f't = {k*3} min'); return [im, ttl]
anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=400)
plt.close(fig)
HTML(anim.to_jshtml())


## 次のステップ

- **[チュートリアル](https://github.com/ENCflow/ENCflow/blob/main/docs/tutorial.md)** —
  実地形の流域計算まで段階的に進みます。
- **[用途集](https://github.com/ENCflow/ENCflow/blob/main/docs/users_guide/usecases.md)** —
  計算したい現象(ため池決壊・内水氾濫・土石流…)から設定を引けます。
- **自分のパソコンで動かす** — Windows の方は
  [Windows での使い方](https://github.com/ENCflow/ENCflow/blob/main/docs/windows.md) へ。
  このノートブックで使ったコマンドがそのまま通用します。

注意: Colab のセッションは切断されるとファイルが消えます。作った結果を
残したいときは、左のファイルペインからダウンロードしてください。
